# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the [FAIR^2 Clinical Colorectal Cancer Survivors dataset](https://sen.science/doi/10.71728/senscience.qs2f-h81p) using the `mlcroissant` library.

### Dataset Source
The dataset is defined via a Croissant schema hosted at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed. Uncomment below if needed.
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Version: {metadata.version}\nPublished on: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their IDs (all referenced by `@id`).

In [ ]:
# List all record sets, their field @id's, and possible column/field details
record_set_infos = []
for rs in dataset.record_sets:
    rec = {
        'name': getattr(rs, 'name', None),
        '@id': getattr(rs, '@id', None),
        'fields': [getattr(f, '@id', None) for f in getattr(rs, 'fields', [])]
    }
    record_set_infos.append(rec)

print("Available record sets and their fields:")
for info in record_set_infos:
    print(f"RecordSet name: {info['name']}, @id: {info['@id']}")
    print(f"  Fields @id's: {info['fields']}")

### Inspect Example Records
Using the first record set as an example, display the first record for exploration.

In [ ]:
# Get the @id of the first record set
if record_set_infos:
    first_rs_id = record_set_infos[0]['@id']
    print(f"Example records for RecordSet @id: {first_rs_id}\n")
    ex_records_iterator = dataset.records(record_set=first_rs_id)
    for i, record in enumerate(ex_records_iterator):
        if i >= 3:
            break
        print(record)
else:
    print("No record sets found in the dataset.")

## 3. Data Extraction
Load data from all record sets into DataFrames for analysis. Field and record set references are always by `@id` in code and explanations.

In [ ]:
# Collect all record set @id's
record_set_ids = [info['@id'] for info in record_set_infos]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for RecordSet @id: {record_set_id}")

# Show the columns of the first record set
if record_set_ids:
    main_rs_id = record_set_ids[0]
    print("\nSample columns in DataFrame for @id:", main_rs_id)
    print(dataframes[main_rs_id].columns.tolist())
    dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Explore a numeric field. We will find and select a numeric field by its `@id` and perform basic normalization and grouping. Modify the `numeric_field_id` and `group_field_id` variables below as desired.


In [ ]:
# Examine columns and infer a numeric and group field by their @id (use dataset's documentation if unsure)
df = dataframes[main_rs_id]
print("Columns available for analysis:")
for col in df.columns:
    print(col)

# For illustration, let's try to select age at second diagnosis as a numeric field (if it exists)
# Otherwise, pick a numeric field present in the dataset -- replace these @id values if needed
candidate_numeric_ids = [col for col in df.columns if 'age' in col.lower() or 'interval' in col.lower() or 'years' in col.lower() or df[col].dtype.kind in ['i', 'f']]
if not candidate_numeric_ids:
    candidate_numeric_ids = [col for col in df.columns if df[col].dtype.kind in ['i', 'f']]
if candidate_numeric_ids:
    numeric_field_id = candidate_numeric_ids[0]
    print(f"Using numeric field with @id: {numeric_field_id}")
else:
    print("No obvious numeric field found.")

candidate_group_ids = [col for col in df.columns if 'sex' in col.lower() or 'gender' in col.lower() or 'site' in col.lower() or 'location' in col.lower() or df[col].dtype == object]
group_field_id = candidate_group_ids[0] if candidate_group_ids else None
if group_field_id:
    print(f"Grouping by field with @id: {group_field_id}")

# Filter, normalize, and group data
if numeric_field_id in df.columns:
    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id}:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Group if possible
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())

## 5. Visualization
Here, let's visualize the distribution of the numeric field (raw and normalized), and show a boxplot if grouping is possible.

In [ ]:
if numeric_field_id in df.columns:
    plt.figure(figsize=(12,5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()
    
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(12,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

- The FAIR^2 dataset provides rich clinicopathological data for survivors of second primary colorectal cancer.
- Using `mlcroissant`, we extracted metadata, fields (all referenced by their `@id`), and loaded records into pandas DataFrames for analysis.
- We explored numeric and grouping fields, normalized distributions, and visualized important relationships.
- This approach is extensible to additional fields and custom analyses as required for your research or application.

**All references to entities within the dataset (record sets, fields, columns) are consistently via their `@id` throughout this notebook.**